# Run the nycped analysis on Colab

This notebook mounts Drive, installs the package and runs the commands in the README. Data files live in `MyDrive/TDSP/nyc_walks`.

While the repository is private, add a GitHub access credential as a Colab secret named `GITHUB_TOKEN` (key icon in the left panel). Do not paste credentials into a cell.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/TDSP/nyc_walks'
os.makedirs(DATA, exist_ok=True)
os.environ['NYCPED_DATA_DIR'] = DATA

In [ ]:
# Install the package from GitHub. A credential is needed only while the repository is private.
try:
    from google.colab import userdata
    key = userdata.get('GITHUB_TOKEN')
except Exception:
    key = None
url = 'git+https://' + (key + '@' if key else '') + 'github.com/AdiBak/nyc-pedestrian-poverty-exposure.git'
!pip install -q "$url"
!python -m nycped --help

## 1. Download the public sources

Writes `MANIFEST.json` with the URL, date, size and checksum of each file.

In [ ]:
!python -m nycped fetch

## 2. Check the numbers from the saved site table

The file `sites_eps25_r30_by_year.csv` is in your data folder from the earlier runs. All six checks should print PASS.

In [ ]:
!python -m nycped verify --sites "$DATA/sites_eps25_r30_by_year.csv"

## 3. Rebuild the site table from the raw sources

Needs `tract_table.csv` and `ped_harm_crashes.csv`. Create them by running `scripts/00_export_processed_inputs.py` in the TDSP notebook session after the Milestone 5 setup. This cell takes a few minutes.

In [ ]:
!python -m nycped sites --crashes "$DATA/ped_harm_crashes.csv" --tract-table "$DATA/tract_table.csv"

## 4. Models, robustness checks and scale analysis

In [ ]:
!python -m nycped models --sites "$DATA/site_dataset.csv" --bootstrap 500

In [ ]:
!python -m nycped robustness --sites "$DATA/site_dataset.csv"

In [ ]:
!python -m nycped scale --sites "$DATA/site_dataset.csv"